# Notebook 07 — Baseline Model Comparison

Trains and evaluates five baseline models on the same dataset and test split as the main fusion system, providing a direct performance comparison to 
justify each architectural decision.

## What This Notebook Does
- Loads the diverse dataset and feature matrices
- Trains five baseline classifiers on the same 70/15/15 split
- Evaluates all five using the same six metrics as the main system
- Generates baseline comparison table and graphs
- Computes AUC-ROC for all baselines including TF-IDF models

## The Five Baselines

### Baseline 1 — Zero-shot Qwen2.5 3B
Uses zero-shot LLM confidence scores without any fine-tuning. Demonstrates that unvalidated LLMs are insufficient for AI phishing detection.

### Baseline 2 — Stylometric Only + XGBoost
Uses only the 63 handcrafted stylometric features — no LLM module. Tests whether the LLM module adds value over pure feature engineering.

### Baseline 3 — TF-IDF + Random Forest
Classic NLP baseline converting email text to word frequency vectors, classified by a Random Forest ensemble. Widely used in phishing detection literature.

### Baseline 4 — TF-IDF + Logistic Regression
Same TF-IDF representation with a linear Logistic Regression classifier. Establishes minimum performance for content-based approaches.

### Baseline 5 — ModernBERT Standalone
Fine-tuned ModernBERT predictions used directly without stylometric fusion. Tests whether stylometric features add value on top of the fine-tuned transformer.

## Inputs
- data/processed/stylometric_features_diverse.csv (from Notebook 04/09)
- data/processed/modernbert_diverse_features.csv (from Notebook 05b)
- data/processed/llm_features.csv (from Notebook 05)
- data/processed/dataset_diverse.csv (from Notebook 09)

## Outputs
- Baseline comparison table 
- results/baseline_comparison.png

## Key Finding
Our full fusion system achieves the highest Macro F1 (0.9879) and lowest FPR (0.0117) of all systems, proving every design decision added value.

## Runtime
Approximately 5-10 minutes

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Load dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_final.csv")
print(f"Dataset loaded: {len(dataset)} emails")
print(dataset['label'].value_counts().sort_index())

# Load stylometric features
stylo = pd.read_csv(DATA_PROCESSED / "stylometric_features_final.csv")
print(f"\nStylometric features: {stylo.shape}")

# Same split as all other notebooks
X_text = dataset['text']
y = dataset['label']

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text, y, test_size=0.30, random_state=42, stratify=y)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Same split for stylometric features
X_stylo = stylo.drop(columns=['label'])
X_train_stylo = X_stylo.iloc[y_train.index]
X_test_stylo = X_stylo.iloc[y_test.index]

print(f"\nSplits: Train={len(y_train)}, Val={len(y_val)}, Test={len(y_test)}")

In [ ]:
#Baseline 1: TF-IDF + Logistic Regression
print("Training Baseline 1: TF-IDF + Logistic Regression")

# TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), 
                         stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Train
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_tfidf, y_train)

# Evaluate
lr_preds = lr.predict(X_test_tfidf)
print("\nBaseline 1 — TF-IDF + Logistic Regression:")
print(classification_report(y_test, lr_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))
print(f"Macro F1: {f1_score(y_test, lr_preds, average='macro'):.4f}")

# Save
with open(MODELS_DIR / "baseline_lr.pkl", 'wb') as f:
    pickle.dump((tfidf, lr), f)
print("Saved.")

In [ ]:
#Baseline 2: TF-IDF + Random Forest
print("Training Baseline 2: TF-IDF + Random Forest")

rf = RandomForestClassifier(n_estimators=200, random_state=42, 
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train_tfidf, y_train)

rf_preds = rf.predict(X_test_tfidf)
print("\nBaseline 2 — TF-IDF + Random Forest:")
print(classification_report(y_test, rf_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))
print(f"Macro F1: {f1_score(y_test, rf_preds, average='macro'):.4f}")

with open(MODELS_DIR / "baseline_rf.pkl", 'wb') as f:
    pickle.dump(rf, f)
print("Saved.")

In [ ]:
#Baseline 3: Stylometric Only + XGBoost (Ablation)
print("Training Baseline 3: Stylometric Only + XGBoost")

# Use only stylometric features, no LLM scores
X_train_s, X_temp_s, y_train_s, y_temp_s = train_test_split(
    X_stylo, y, test_size=0.30, random_state=42, stratify=y)
X_val_s, X_test_s, y_val_s, y_test_s = train_test_split(
    X_temp_s, y_temp_s, test_size=0.50, random_state=42, stratify=y_temp_s)

class_counts = y_train_s.value_counts().sort_index()
total = len(y_train_s)
class_weights = {cls: total / (len(class_counts) * count)
                 for cls, count in class_counts.items()}
sample_weights = y_train_s.map(class_weights)

xgb_stylo = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='mlogloss', verbosity=0
)
xgb_stylo.fit(X_train_s, y_train_s, sample_weight=sample_weights)

stylo_preds = xgb_stylo.predict(X_test_s)
print("\nBaseline 3 — Stylometric Only + XGBoost:")
print(classification_report(y_test_s, stylo_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))
print(f"Macro F1: {f1_score(y_test_s, stylo_preds, average='macro'):.4f}")

with open(MODELS_DIR / "baseline_stylo_only.pkl", 'wb') as f:
    pickle.dump(xgb_stylo, f)
print("Saved.")

In [ ]:
 #Baseline 4: Zero-Shot LLM Only (Ablation)
print("Training Baseline 4: Zero-Shot LLM Only")

# Use only zero-shot Qwen2.5 confidence scores
llm = pd.read_csv(DATA_PROCESSED / "llm_features.csv")
llm_only = llm[['conf_legitimate', 'conf_human_phishing', 'conf_ai_phishing']]

X_train_l, X_temp_l, y_train_l, y_temp_l = train_test_split(
    llm_only, y, test_size=0.30, random_state=42, stratify=y)
X_val_l, X_test_l, y_val_l, y_test_l = train_test_split(
    X_temp_l, y_temp_l, test_size=0.50, random_state=42, stratify=y_temp_l)

# Direct prediction from LLM label (no XGBoost needed)
llm_direct_preds = llm['llm_label'].map({
    'legitimate': 0,
    'human_phishing': 1,
    'ai_generated_phishing': 2
}).iloc[y_test_l.index]

print("\nBaseline 4 — Zero-Shot LLM Only (Qwen2.5 3B):")
print(classification_report(y_test_l, llm_direct_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))
print(f"Macro F1: {f1_score(y_test_l, llm_direct_preds, average='macro'):.4f}")
print("No model saved — this uses direct LLM predictions")

In [ ]:
#Baseline 5: ModernBERT Standalone (Ablation)
print("Training Baseline 5: ModernBERT Standalone")

# Use only ModernBERT predictions without stylometric features
modernbert = pd.read_csv(DATA_PROCESSED / "modernbert_features.csv")

X_train_mb, X_temp_mb, y_train_mb, y_temp_mb = train_test_split(
    modernbert[['conf_legitimate','conf_human_phishing','conf_ai_phishing']],
    y, test_size=0.30, random_state=42, stratify=y)
X_val_mb, X_test_mb, y_val_mb, y_test_mb = train_test_split(
    X_temp_mb, y_temp_mb, test_size=0.50, random_state=42, stratify=y_temp_mb)

# Direct prediction from ModernBERT confidence scores
modernbert_preds = modernbert['llm_label'].iloc[y_test_mb.index]

print("\nBaseline 5 — ModernBERT Standalone:")
print(classification_report(y_test_mb, modernbert_preds,
      target_names=['Legitimate', 'Human Phishing', 'AI Phishing']))
print(f"Macro F1: {f1_score(y_test_mb, modernbert_preds, average='macro'):.4f}")
print("No model saved — this uses direct ModernBERT predictions")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Load dataset
dataset = pd.read_csv(DATA_PROCESSED / "dataset_diverse.csv")
X_text = dataset['text']
y = dataset['label']

# Same split
X_train_text, X_temp, y_train, y_temp = train_test_split(
    X_text, y, test_size=0.30, random_state=42, stratify=y)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# TF-IDF
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_tfidf, y_train)
lr_proba = lr.predict_proba(X_test_tfidf)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42,
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
rf_proba = rf.predict_proba(X_test_tfidf)

# Compute AUC-ROC
y_test_bin = label_binarize(y_test, classes=[0,1,2])
lr_auc = roc_auc_score(y_test_bin, lr_proba, multi_class='ovr', average='macro')
rf_auc = roc_auc_score(y_test_bin, rf_proba, multi_class='ovr', average='macro')

print(f"TF-IDF + Logistic Regression AUC-ROC: {lr_auc:.4f}")
print(f"TF-IDF + Random Forest AUC-ROC:       {rf_auc:.4f}")